# LSTM (Bird Collision Prediction)

#### LSTM Model Training


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Reshape
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --- Configuration and Data Loading ---
video_output_dir = "D:/Project_P1/results/video_analysis"
csv_path = "bird_sequences.csv"

# Ensure output dir exists
os.makedirs(video_output_dir, exist_ok=True)

try:
    df_detections = pd.read_csv(csv_path)
    if df_detections.empty:
        raise ValueError("The CSV file is empty. Please ensure it contains data.")
except FileNotFoundError:
    print(f"Error: The file '{csv_path}' was not found.")
    exit()
except ValueError as e:
    print(f"Data error: {e}")
    exit()

# --- Calculate center points ---
df_detections["center_x"] = (df_detections["x1"] + df_detections["x2"]) / 2
df_detections["center_y"] = (df_detections["y1"] + df_detections["y2"]) / 2

# --- Data Preparation Function ---
def prepare_prediction_data(df, sequence_length=15, prediction_steps=5):
    """
    Prepare sequences for all videos.
    Returns X, y arrays concatenated across all videos.
    """
    X_all, y_all = [], []

    for video_id, group in df.groupby("video"):
        group = group.sort_values("frame").reset_index(drop=True)
        coordinates = group[["center_x", "center_y"]].values

        for i in range(len(coordinates) - sequence_length - prediction_steps + 1):
            X_all.append(coordinates[i:i + sequence_length])
            y_all.append(coordinates[i + sequence_length:i + sequence_length + prediction_steps])

    return np.array(X_all), np.array(y_all)

# --- Model Creation Function ---
def create_prediction_model(input_shape, output_steps):
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.3),
        LSTM(32),
        Dropout(0.3),
        Dense(32, activation="relu"),
        Dense(output_steps * 2),
        Reshape((output_steps, 2))
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss="mse", metrics=["mae"])
    return model

# --- Prepare Data ---
sequence_length = 15
prediction_steps = 5
X, y = prepare_prediction_data(df_detections, sequence_length, prediction_steps)

if len(X) == 0:
    print("Not enough data to create sequences. Please check your dataset.")
    exit()

# Train/test split
split_idx = int(0.8 * len(X))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

# --- Train Model ---
input_shape = (X_train.shape[1], X_train.shape[2])
model = create_prediction_model(input_shape, prediction_steps)

history = model.fit(
    X_train, y_train,
    batch_size=16,
    epochs=100,
    validation_data=(X_test, y_test),
    verbose=1
)

# --- Predict ---
y_pred = model.predict(X_test)

# --- Evaluation ---
def evaluate_predictions(y_true, y_pred, prediction_steps):
    results = []
    for step in range(prediction_steps):
        mse = mean_squared_error(y_true[:, step, :], y_pred[:, step, :])
        mae = mean_absolute_error(y_true[:, step, :], y_pred[:, step, :])
        distance_errors = np.sqrt(np.sum((y_true[:, step, :] - y_pred[:, step, :])**2, axis=1))
        avg_distance_error = np.mean(distance_errors)

        results.append({
            "prediction_step": step + 1,
            "mse": mse,
            "mae": mae,
            "avg_distance_error": avg_distance_error,
            "std_distance_error": np.std(distance_errors)
        })
    return pd.DataFrame(results)

evaluation_results = evaluate_predictions(y_test, y_pred, prediction_steps)
print("\n=== PREDICTION ACCURACY ===")
print(evaluation_results.to_string(index=False))

# Save model + results
model.save(os.path.join(video_output_dir, "bird_movement_predictor.h5"))
evaluation_results.to_csv(os.path.join(video_output_dir, "prediction_accuracy.csv"), index=False)

print(f"\nModel and results saved to: {video_output_dir}")

# --- Visualization ---
def visualize_single_prediction_side_by_side(X_test, y_test, y_pred, example_idx=0):
    if example_idx >= len(X_test):
        print(f"Error: example_idx {example_idx} is out of bounds for {len(X_test)} samples.")
        return

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    fig.suptitle("Bird Flight Path Prediction Comparison", fontsize=18, y=1.02)

    past_positions = X_test[example_idx]
    actual_future = y_test[example_idx]
    predicted_future = y_pred[example_idx]

    all_x = np.concatenate([past_positions[:, 0], actual_future[:, 0], predicted_future[:, 0]])
    all_y = np.concatenate([past_positions[:, 1], actual_future[:, 1], predicted_future[:, 1]])

    x_min, x_max = all_x.min() - 50, all_x.max() + 50
    y_min, y_max = all_y.min() - 50, all_y.max() + 50

    # Left: Actual
    ax_left = axes[0]
    ax_left.plot(past_positions[:, 0], past_positions[:, 1], "bo-", label="Past positions", alpha=0.7)
    ax_left.plot(actual_future[:, 0], actual_future[:, 1], "go-", label="Actual future", alpha=0.7)
    ax_left.plot([past_positions[-1, 0], actual_future[0, 0]],
                 [past_positions[-1, 1], actual_future[0, 1]], "g--", alpha=0.5)
    ax_left.set_title(f"Actual Bird Path (Example {example_idx+1})")
    ax_left.set_xlabel("X Position")
    ax_left.set_ylabel("Y Position")
    ax_left.legend()
    ax_left.grid(True, alpha=0.3)
    ax_left.set_xlim(x_min, x_max)
    ax_left.set_ylim(y_min, y_max)
    ax_left.invert_yaxis()

    # Right: Predicted
    ax_right = axes[1]
    ax_right.plot(past_positions[:, 0], past_positions[:, 1], "bo-", label="Past positions", alpha=0.7)
    ax_right.plot(predicted_future[:, 0], predicted_future[:, 1], "ro-", label="Predicted future", alpha=0.7)
    ax_right.plot([past_positions[-1, 0], predicted_future[0, 0]],
                  [past_positions[-1, 1], predicted_future[0, 1]], "r--", alpha=0.5)
    ax_right.set_title(f"Predicted Bird Path (Example {example_idx+1})")
    ax_right.set_xlabel("X Position")
    ax_right.set_ylabel("Y Position")
    ax_right.legend()
    ax_right.grid(True, alpha=0.3)
    ax_right.set_xlim(x_min, x_max)
    ax_right.set_ylim(y_min, y_max)
    ax_right.invert_yaxis()

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(os.path.join(video_output_dir, f"prediction_comparison_example_{example_idx}.png"),
                dpi=300, bbox_inches="tight")
    plt.show()

visualize_single_prediction_side_by_side(X_test, y_test, y_pred, example_idx=0)
print("Prediction comparison plot generated.")


#### Using Tflite File for real time processing

In [2]:
import os
import cv2
import pandas as pd
import numpy as np
import tensorflow as tf
from collections import deque
import time

# --- 1. Configuration ---

# CORE PATHS
VIDEO_PATH = "D:/Project_P1/data/valid/test_video.mp4"
DETECTIONS_CSV_PATH = "D:/Project_P1/results/video_analysis/bird_detections.csv"
# --- IMPORTANT: This now points to your optimized TFLite model ---
TFLITE_MODEL_PATH = "D:/Project_P1/models/model.tflite"
OUTPUT_VIDEO_PATH = "D:/Project_P1/results/video_analysis/video_with_tflite_predictions.mp4"

# MODEL PARAMETERS (should match the trained model)
SEQUENCE_LENGTH = 15
PREDICTION_STEPS = 5

# DYNAMIC ADAPTATION PARAMETERS
# Defines how many frames to skip based on the bird's movement pattern.
# A lower number means more frequent processing.
PATTERN_TO_INTERVAL_MAP = {
    "Steady flight": 5,   # Process 1 in 5 frames for predictable movement
    "Maneuvering": 2,     # Process 1 in 2 frames for complex movement
    "Erratic flight": 2,  # Process 1 in 2 frames for unpredictable movement
    "Unknown": 3          # A default interval for other cases
}
DEFAULT_INTERVAL = 3 # Fallback if no sequence is identified

# VISUALIZATION PARAMETERS
ARROW_COLOR = (0, 255, 0)
ARROW_THICKNESS = 2
PATH_COLOR = (255, 0, 0)
PATH_THICKNESS = 2

# --- 2. Data Analysis and Flight Pattern Recognition (Pre-computation) ---

def identify_sequences(df, frame_gap_threshold=15):
    """Identify discrete movement sequences based on gaps between detection frames."""
    df_sorted = df.sort_values('frame')
    frame_diffs = df_sorted['frame'].diff()
    sequence_breaks = (frame_diffs > frame_gap_threshold).cumsum()
    df_sorted['sequence_id'] = sequence_breaks
    return df_sorted

def analyze_flight_patterns(df_with_sequences):
    """Analyze kinematics of each sequence to classify the flight pattern."""
    patterns = []
    for seq_id, seq_data in df_with_sequences.groupby('sequence_id'):
        if len(seq_data) < 5:  # Need enough data points for a reliable analysis
            continue
            
        seq_data = seq_data.sort_values('frame').copy()
        
        # Calculate derivatives (velocity and acceleration)
        x, y, frames = seq_data['center_x'].values, seq_data['center_y'].values, seq_data['frame'].values
        
        dx, dy = np.gradient(x, frames), np.gradient(y, frames)
        velocity = np.sqrt(dx**2 + dy**2)
        
        ddx, ddy = np.gradient(dx, frames), np.gradient(dy, frames)
        acceleration = np.sqrt(ddx**2 + ddy**2)
        
        # Classify the pattern based on statistical thresholds
        avg_vel, std_vel, avg_acc = np.mean(velocity), np.std(velocity), np.mean(acceleration)
        
        pattern_type = "Unknown"
        if std_vel < 5 and avg_acc < 2: pattern_type = "Steady flight"
        elif avg_acc > 5: pattern_type = "Maneuvering"
        elif std_vel > 10: pattern_type = "Erratic flight"
        
        patterns.append({
            'sequence_id': seq_id, 'pattern_type': pattern_type, 
            'avg_velocity': avg_vel, 'velocity_std': std_vel, 
            'avg_acceleration': avg_acc
        })
    
    return pd.DataFrame(patterns) if patterns else pd.DataFrame()

print("--- Phase 1: Pre-Analysis of Bird Detections ---")
try:
    df_detections = pd.read_csv(DETECTIONS_CSV_PATH)
    df_detections['center_x'] = (df_detections['x1'] + df_detections['x2']) / 2
    df_detections['center_y'] = (df_detections['y1'] + df_detections['y2']) / 2
except FileNotFoundError:
    print(f"ERROR: Detections file not found at {DETECTIONS_CSV_PATH}")
    exit()

df_with_sequences = identify_sequences(df_detections)
print(f"Identified {df_with_sequences['sequence_id'].nunique()} unique movement sequences.")

df_patterns = analyze_flight_patterns(df_with_sequences)
if not df_patterns.empty:
    print("\n=== FLIGHT PATTERN ANALYSIS SUMMARY ===")
    print(df_patterns.to_string(index=False))
else:
    print("Not enough data for flight pattern analysis. Will use default settings.")

# --- 3. Setup Video Processing with TFLite and Dynamic Strategy ---

print("\n--- Phase 2: Setting up Video Processing with TFLite ---")
# Load the TFLite model and allocate memory (tensors)
try:
    interpreter = tf.lite.Interpreter(model_path=TFLITE_MODEL_PATH)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    print("TFLite model loaded successfully.")
except Exception as e:
    print(f"ERROR: Failed to load TFLite model at {TFLITE_MODEL_PATH}.")
    print(f"Error details: {e}")
    exit()

# Open video file and get properties
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    print(f"ERROR: Could not open video file at {VIDEO_PATH}")
    exit()

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Setup video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (frame_width, frame_height))

# Create a quick lookup dictionary for detections by frame number
detections_by_frame = {frame: group[['center_x', 'center_y']].to_numpy() 
                       for frame, group in df_detections.groupby('frame')}

print("Creating dynamic frame processing strategy...")
# Create a map to store the processing interval for each frame of the video
frame_to_interval_map = np.full(total_frames, DEFAULT_INTERVAL, dtype=int)
if not df_patterns.empty:
    df_sequences_with_patterns = pd.merge(df_with_sequences, df_patterns, on='sequence_id', how='left')
    for _, row in df_sequences_with_patterns.iterrows():
        # Find the start and end frame of the current sequence
        sequence_frames = df_with_sequences[df_with_sequences['sequence_id'] == row['sequence_id']]['frame']
        start_frame, end_frame = int(sequence_frames.min()), int(sequence_frames.max())
        
        interval = PATTERN_TO_INTERVAL_MAP.get(row['pattern_type'], DEFAULT_INTERVAL)
        # Assign the calculated interval to all frames in this sequence
        if start_frame < len(frame_to_interval_map) and end_frame < len(frame_to_interval_map):
            frame_to_interval_map[start_frame : end_frame + 1] = interval

print("Strategy created. Ready for processing.")

# --- 4. Process Video with Dynamic Frame Selection ---

print("\n--- Phase 3: Processing Video with Dynamic Adaptation ---")
path_history = deque(maxlen=SEQUENCE_LENGTH)
frame_number = 0
prediction_times = []
last_drawn_path_points = None
last_drawn_arrow = None

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    # Get the dynamically determined processing interval for this specific frame
    current_interval = frame_to_interval_map[frame_number]
    
    # CORE LOGIC: Only run prediction on frames selected by our dynamic strategy
    if frame_number % current_interval == 0:
        if frame_number in detections_by_frame:
            current_position = detections_by_frame[frame_number][0]
            path_history.append(current_position)

            if len(path_history) > 1:
                last_drawn_path_points = np.array(path_history, dtype=np.int32).reshape((-1, 1, 2))

            if len(path_history) == SEQUENCE_LENGTH:
                # Prepare input data, ensuring it's the correct type (float32)
                input_sequence = np.array(path_history, dtype=np.float32).reshape(1, SEQUENCE_LENGTH, 2)
                
                start_time = time.perf_counter()
                
                # Use the TFLite interpreter for prediction
                interpreter.set_tensor(input_details[0]['index'], input_sequence)
                interpreter.invoke() # Run inference
                predicted_path = interpreter.get_tensor(output_details[0]['index'])
                
                end_time = time.perf_counter()
                prediction_times.append(end_time - start_time)
                
                # Store the arrow coordinates to be drawn on every frame
                predicted_next_point = tuple(predicted_path[0, 0, :].astype(int))
                current_pos_int = tuple(current_position.astype(int))
                last_drawn_arrow = (current_pos_int, predicted_next_point)
        else:
            # If no detection, clear the history and visuals to start a fresh track
            path_history.clear()
            last_drawn_path_points = None
            last_drawn_arrow = None

    # VISUALIZATION: Draw on EVERY frame for a smooth video output
    if last_drawn_path_points is not None:
        cv2.polylines(frame, [last_drawn_path_points], isClosed=False, color=PATH_COLOR, thickness=PATH_THICKNESS)
    
    if last_drawn_arrow is not None:
        cv2.arrowedLine(frame, last_drawn_arrow[0], last_drawn_arrow[1], ARROW_COLOR, ARROW_THICKNESS, tipLength=0.3)

    out.write(frame)
    
    if frame_number > 0 and frame_number % 100 == 0:
        print(f"   Processed frame {frame_number}/{total_frames}")

    frame_number += 1

# --- 5. Finalize and Report ---
print("\n--- Phase 4: Finalizing and Reporting ---")
cap.release()
out.release()
cv2.destroyAllWindows()
print(f"\n✅ Success! Video with dynamic predictions saved to:\n{OUTPUT_VIDEO_PATH}")

if prediction_times:
    avg_time_ms = np.mean(prediction_times) * 1000
    time_budget_ms = (1 / fps) * 1000
    
    print("\n--- TFLite Model Performance (on processed frames) ---")
    print(f"Video FPS: {fps} (Time budget per frame: {time_budget_ms:.2f} ms)")
    print("-----------------------------------------------------")
    print(f"Total Predictions Made: {len(prediction_times)}")
    print(f"Average Prediction Time: {avg_time_ms:.2f} ms")
    print(f"Maximum Prediction Time: {np.max(prediction_times) * 1000:.2f} ms")
    print(f"Minimum Prediction Time: {np.min(prediction_times) * 1000:.2f} ms")
    print("-----------------------------------------------------\n")
    if avg_time_ms < time_budget_ms:
        print("🟢 Analysis: The model's prediction speed is sufficient for real-time processing.")
    else:
        print("🔴 Analysis: The model's prediction speed may still be a bottleneck. Consider more aggressive frame skipping or hardware acceleration.")
else:
    print("\nNo predictions were made during video processing.")

--- Phase 1: Pre-Analysis of Bird Detections ---
Identified 1 unique movement sequences.

=== FLIGHT PATTERN ANALYSIS SUMMARY ===
 sequence_id  pattern_type  avg_velocity  velocity_std  avg_acceleration
           0 Steady flight       1.39215      1.110501          0.378283

--- Phase 2: Setting up Video Processing with TFLite ---
TFLite model loaded successfully.
Creating dynamic frame processing strategy...
Strategy created. Ready for processing.

--- Phase 3: Processing Video with Dynamic Adaptation ---
   Processed frame 100/390
   Processed frame 200/390
   Processed frame 300/390

--- Phase 4: Finalizing and Reporting ---

✅ Success! Video with dynamic predictions saved to:
D:/Project_P1/results/video_analysis/video_with_tflite_predictions.mp4

--- TFLite Model Performance (on processed frames) ---
Video FPS: 30 (Time budget per frame: 33.33 ms)
-----------------------------------------------------
Total Predictions Made: 64
Average Prediction Time: 0.64 ms
Maximum Prediction Ti

#### Using .h5 file

In [ ]:
import os
import cv2
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from collections import deque

# --- 1. Configuration ---
# INPUTS
VIDEO_PATH = "D:/Project_P1/data/valid/test_video.mp4"
DETECTIONS_CSV_PATH = "D:/Project_P1/results/video_analysis/bird_detections.csv"
MODEL_PATH = "D:/Project_P1/results/lstm_model/bird_movement_predictor.h5"

# OUTPUT
OUTPUT_VIDEO_PATH = "D:/Project_P1/results/video_analysis/video_with_predictions.mp4"

# MODEL PARAMETERS (should match the trained model)
SEQUENCE_LENGTH = 15
PREDICTION_STEPS = 5

# VISUALIZATION PARAMETERS
ARROW_COLOR = (0, 255, 0)  # Green
ARROW_THICKNESS = 2
PATH_COLOR = (255, 0, 0) # Blue
PATH_THICKNESS = 2

# --- 2. Load Model and Data ---
print("Loading model and detection data...")
# Load the pre-trained model for position prediction
model = load_model(MODEL_PATH, compile=False)

# Load the detection data and prepare it for quick lookups
try:
    df_detections = pd.read_csv(DETECTIONS_CSV_PATH)
    df_detections['center_x'] = (df_detections['x1'] + df_detections['x2']) / 2
    df_detections['center_y'] = (df_detections['y1'] + df_detections['y2']) / 2
    
    # Group detections by frame number for efficient access
    detections_by_frame = {frame: group[['center_x', 'center_y']].to_numpy() 
                           for frame, group in df_detections.groupby('frame')}
except FileNotFoundError:
    print(f"ERROR: Detections file not found at {DETECTIONS_CSV_PATH}")
    exit()

# --- 3. Setup Video Processing ---
print("Setting up video processing...")
# Open the source video
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    print(f"ERROR: Could not open video file at {VIDEO_PATH}")
    exit()

# Get video properties for the output file
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Define the codec and create VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (frame_width, frame_height))

# --- 4. Process Video Frame by Frame ---
# Use a deque to store the recent path history of the bird
# A deque is like a list but with fast appends and pops from both ends
path_history = deque(maxlen=SEQUENCE_LENGTH)
frame_number = 0

print("Processing video and making predictions...")
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break # End of video

    # Get the bird's position for the current frame
    if frame_number in detections_by_frame:
        # Assuming only one bird for simplicity, take the first detection
        current_position = detections_by_frame[frame_number][0]
        current_pos_int = tuple(current_position.astype(int))
        
        # Add the current position to our history
        path_history.append(current_position)

        # Draw the recent path of the bird
        if len(path_history) > 1:
            points = np.array(path_history, dtype=np.int32).reshape((-1, 1, 2))
            cv2.polylines(frame, [points], isClosed=False, color=PATH_COLOR, thickness=PATH_THICKNESS)

        # Check if we have enough history to make a prediction
        if len(path_history) == SEQUENCE_LENGTH:
            # Prepare the sequence for the model
            input_sequence = np.array(path_history).reshape(1, SEQUENCE_LENGTH, 2)
            
            # Predict the future path
            predicted_path = model.predict(input_sequence) # Shape: (1, 5, 2)
            
            # Get the first predicted point as the arrow's destination
            predicted_next_point = predicted_path[0, 0, :].astype(int)
            
            # Draw the prediction arrow
            cv2.arrowedLine(frame, 
                            current_pos_int, 
                            tuple(predicted_next_point), 
                            ARROW_COLOR, 
                            ARROW_THICKNESS,
                            tipLength=0.3)
    else:
        # If no detection on this frame, clear the history to start a new track
        path_history.clear()

    # Write the annotated frame to the output video
    out.write(frame)
    
    # Optional: Display progress
    if frame_number % 100 == 0:
        print(f"  Processed frame {frame_number}/{total_frames}")

    frame_number += 1

# --- 5. Finalize ---
print("Finalizing video...")
cap.release()
out.release()
cv2.destroyAllWindows()
print(f"\n✅ Success! Video with prediction arrows saved to:\n{OUTPUT_VIDEO_PATH}")

Loading model and detection data...
Setting up video processing...
Processing video and making predictions...
  Processed frame 0/390
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 355ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 5